<a href="https://colab.research.google.com/github/gabrielMorais21/tech_challenge_3/blob/main/tech_challenge_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Tech Challenge - 3

Esta célula é responsável pela instalação de todas as bibliotecas necessárias e organiza as importações para uma melhor legibilidade e manutenção do código. Ela inclui bibliotecas para processamento de PDFs (pypdf, langchain), interação com LLMs (langchain-groq, transformers), manipulação de dados e treino (datasets, peft, trl, bitsandbytes, accelerate), e integração com o Hugging Face (huggingface_hub)

In [3]:
# ==========================================
# 1. INSTALAÇÃO DE BIBLIOTECAS
# ==========================================
!pip install -q -U pypdf langchain langchain-community langchain-groq transformers datasets peft trl bitsandbytes accelerate huggingface_hub

# ==========================================
# 2. IMPORTAÇÕES ORGANIZADAS
# ==========================================
# Bibliotecas Nativas do Python e PyTorch
import os
import json
import time
import torch

# Google Colab (Para puxar senhas/tokens)
from google.colab import userdata

# Ecossistema LangChain (Para o Web Scraping, PDF e Agentes)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Ecossistema Hugging Face (Para o Fine-Tuning e Llama 3)
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

/tmp/ipykernel_32917/1922656287.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Esta célula é responsável por carregar um arquivo PDF local e dividir o seu conteúdo em fragmentos menores e gerenciáveis. Ela utiliza o `PyPDFLoader` para carregar o PDF e o `RecursiveCharacterTextSplitter` para dividir o texto, garantindo que cada fragmento tenha no máximo 1000 caracteres com uma sobreposição de 100 caracteres entre os fragmentos para manter o contexto.

In [15]:
# 1. Carrega o PDF local
loader = PyPDFLoader("/content/Manejo_clinico_Dengue.pdf")
documentos = loader.load()

# 2. Divide o texto em blocos menores ⚙️
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
fragmentos = text_splitter.split_documents(documentos)

print(f"O PDF foi dividido em {len(fragmentos)} blocos de texto.")

O PDF foi dividido em 146 blocos de texto.


Esta célula configura o ambiente para interagir com a API da Groq e define a lógica principal para extrair informações estruturadas de fragmentos de texto. Ela envolve:

1. **Chave da API da Groq**: Recupera a chave da API de forma segura dos dados de usuário do Colab.
2. **Configuração do LLM**: Inicializa o modelo `ChatGroq` (Llama 3.1-8b-instant) para uma geração rápida e eficiente de saída em JSON.
3. **Prompt de Sistema**: Define um prompt de sistema rigoroso para o LLM, instruindo-o a atuar como um especialista médico, extrair informações de fragmentos de protocolo e formatar a saída como um JSON com as chaves `Instrucao`, `Entrada` e `Saida`, incluindo notas de segurança.
4. **Template de Prompt**: Cria um `ChatPromptTemplate` que combina o prompt de sistema com uma entrada do usuário para o texto do protocolo.
5. **Criação da Cadeia (Chain)**: Monta o prompt e o LLM em uma cadeia (chain) do LangChain.
6. **Teste Inicial**: Invoca a cadeia com o primeiro fragmento para demonstrar a sua funcionalidade e imprimir a saída estruturada.

In [16]:
# 1. Puxa a chave da Groq
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# 2. Configura o Llama 3 (Ultra-rápido e excelente para JSON)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)

# 3. Nosso Prompt de Sistema blindado
system_prompt = """Atue como um médico especialista focado em preservar a vida do paciente. Sua tarefa é ler o fragmento de texto de um protocolo médico e criar um exemplo de treinamento.
Regras:
1. NÃO invente condutas médicas. Siga estritamente o que foi especificado no texto fornecido.
2. Tenha um viés conservador de segurança.
3. Formate sua resposta EXATAMENTE como um JSON válido contendo estas três chaves:
- "Instrucao": Uma pergunta de um médico ou cenário clínico pedindo orientação.
- "Entrada": O fragmento de texto exato do protocolo.
- "Saida": A conduta médica detalhada baseada no texto, incluindo notas de segurança.
Retorne APENAS o JSON."""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "Texto do protocolo:\n{texto_fragmento}")
])

# 4. Cria a cadeia (Chain)
chain = prompt_template | llm

# 5. Testa a extração com o primeiro fragmento
resposta = chain.invoke({"texto_fragmento": fragmentos[0].page_content})
print(resposta.content)

{
  "Instrucao": "Um paciente adulto com sintomas de dengue foi encaminhado ao pronto-socorro. Qual é a conduta recomendada para o diagnóstico e manejo clínico?",
  "Entrada": "Adulto e criança DENGUE DIAGNÓSTICO E MANEJO CLÍNICO 6ª edição Brasília DF 2024 VENDA PROIBI DA VENDA PROIBI DADISTRIBUIÇÃO       GRATUITA MINISTÉRIO DA SAÚDE",
  "Saida": {
    "Nota de Segurança": "Antes de iniciar o tratamento, é importante verificar se o paciente tem alguma contraindicação para o manejo clínico da dengue.",
    "Diagnóstico": "O diagnóstico da dengue deve ser baseado em critérios clínicos, laboratoriais e epidemiológicos. É importante coletar amostras de sangue para análise de hemograma e detecção de vírus.",
    "Manejo Clínico": "O manejo clínico da dengue deve ser realizado de acordo com as diretrizes do Ministério da Saúde. Isso inclui a administração de fluidos intravenosos para prevenir a desidratação, a monitorização da pressão arterial e a administração de medicamentos para controlar

Esta célula implementa um mecanismo robusto para extrair dados estruturados de todos os fragmentos do PDF, lidando com potenciais limites de taxa da API ou saídas JSON inválidas. Os principais passos incluem:

1. **JSON Output Parser**: Inicializa um `JsonOutputParser` para garantir que a saída do LLM seja corretamente analisada em um formato JSON.
2. **Cadeia Robusta**: Atualiza a cadeia existente do LangChain para incluir o `JsonOutputParser`.
3. **Processamento Iterativo**: Realiza uma iteração (loop) por cada `fragmento` gerado a partir do PDF.
4. **Tratamento de Erros e Tentativas**: Inclui um bloco `try-except` para capturar exceções, particularmente erros de `Invalid json output`, que frequentemente indicam limites de taxa ou respostas malformadas. Ela introduz chamadas `time.sleep()` para pausar a execução, dando tempo à API para se recuperar.
5. **Geração do Dataset**: Invoca a cadeia robusta para cada fragmento, convertendo a saída para um formato JSONL (JSON Lines) e adicionando-a ao arquivo `dataset_protocolo_dengue.jsonl`.

In [17]:
# 1. Inicializamos o analisador automático
parser = JsonOutputParser()

# 2. Atualizamos a nossa cadeia
chain_robusta = prompt_template | llm | parser

print("A iniciar a extração segura (com pausas para a API)...")

nome_ficheiro = "dataset_protocolo_dengue.jsonl"
dados_treinamento = []

# 3. Processar um a um e guardar imediatamente
with open(nome_ficheiro, "w", encoding="utf-8") as f:
    for i, frag in enumerate(fragmentos):
        print(f"A processar fragmento {i+1} de {len(fragmentos)}...")
        try:
            # Pede ao modelo para analisar apenas este fragmento
            resultado = chain_robusta.invoke({"texto_fragmento": frag.page_content})

            # Se for uma lista de dicionários, guardamos um a um
            if isinstance(resultado, list):
                for item in resultado:
                    f.write(json.dumps(item, ensure_ascii=False) + "\n")
            else:
                f.write(json.dumps(resultado, ensure_ascii=False) + "\n")

            # A pausa mágica de 5 segundos para a API recuperar o fôlego
            time.sleep(5)

        except Exception as e:
            print(f"⚠️ Erro de limite no fragmento {i+1} (A aguardar 15s para recuperar) - Detalhe: {e}")
            # Se batermos no limite, damos uma pausa maior e continuamos
            time.sleep(15)

print(f"\nConcluído com sucesso! Pode verificar o ficheiro '{nome_ficheiro}'.")

A iniciar a extração segura (com pausas para a API)...
A processar fragmento 1 de 146...
A processar fragmento 2 de 146...
A processar fragmento 3 de 146...
A processar fragmento 4 de 146...
A processar fragmento 5 de 146...
⚠️ Erro de limite no fragmento 5 (A aguardar 15s para recuperar) - Detalhe: Invalid json output: {
  "Instrucao": "Um paciente adulto com sintomas de febre alta e dor de cabeça foi encaminhado ao pronto-socorro. Qual é a conduta recomendada para o diagnóstico e manejo clínico da dengue em adultos?",
  "Entrada": "Dengue : diagnóstico e manejo clínico : adulto e criança [recurso eletrônico] / Ministério da Saúde, Secretaria de Vigilância em Saúde e Ambiente, Departamento de Doenças Transmissíveis. – 6. ed. – Brasília : Ministério da Saúde, 2024.",
  "Saida": {
    "Diagnóstico": {
      "Realizar exame de sangue para detecção de IgM e IgG anti-dengue",
      "Coletar história clínica e exame físico para identificar sintomas da dengue",
      "Realizar exame de image

Esta célula é dedicada à autenticação no Hugging Face Hub, o que é essencial para baixar modelos e datasets e, potencialmente, fazer o upload de modelos com fine-tuning. Ela recupera o token do Hugging Face dos dados de usuário do Colab e utiliza o `huggingface_hub.login()` para estabelecer a sessão.

In [18]:


# 2. Fazer o login automático no Hugging Face usando o nosso segredo


hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Login no Hugging Face concluído com sucesso!")

Login no Hugging Face concluído com sucesso!


Esta célula carrega o modelo pré-treinado Llama 3 8B Instruct do Hugging Face (`NousResearch/Meta-Llama-3-8B-Instruct`) com quantização de 4 bits. Isso reduz significativamente o uso de memória, permitindo que o modelo caiba em GPUs com VRAM limitada. Ela também inicializa o tokenizer e ativa o gradient checkpointing para otimizar ainda mais a memória durante o fine-tuning.

In [6]:
nome_modelo = "NousResearch/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("A carregar o modelo otimizado (em 4-bits)...")

tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
tokenizer.pad_token = tokenizer.eos_token

modelo = AutoModelForCausalLM.from_pretrained(
    nome_modelo,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16 # O SEGREDO QUE FALTAVA
)

modelo.gradient_checkpointing_enable()

print("Modelo Llama 3 carregado com sucesso na GPU! 🚀")

A carregar o modelo otimizado (em 4-bits)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Modelo Llama 3 carregado com sucesso na GPU! 🚀


Esta célula prepara o dataset e configura o método LoRA (Low-Rank Adaptation) para o fine-tuning do modelo Llama 3. Ela envolve:

1. **Carregamento do Dataset**: Carrega o arquivo `dataset_protocolo_dengue.jsonl` em um objeto `Dataset` do Hugging Face.
2. **Formatação do Prompt**: Define uma função `formatar_prompt` para estruturar os exemplos do dataset em um formato de prompt específico, adequado para modelos que seguem instruções, incluindo as seções `Instrução`, `Contexto` e `Resposta`.
3. **Configuração do LoRA**: Configura o `LoraConfig` com parâmetros como `r`, `lora_alpha`, `target_modules` e `lora_dropout` para definir como os adaptadores LoRA serão aplicados ao modelo base.
4. **Modelo PEFT**: Aplica a configuração do LoRA ao modelo Llama 3 carregado utilizando `get_peft_model`, criando um modelo PEFT (Parameter-Efficient Fine-Tuning) treinável.
5. **Argumentos de Treino**: Configura os `TrainingArguments` para o `SFTTrainer`, especificando parâmetros como tamanho do lote (batch size), taxa de aprendizado (learning rate), número de épocas e estratégia de otimização.
6. **Inicialização do SFTTrainer**: Inicializa o `SFTTrainer` com o modelo PEFT, o dataset formatado, o tokenizer e os argumentos de treino.
7. **Execução do Treino**: Inicia o processo de fine-tuning usando `trainer.train()`. (Nota: A execução foi interrompida, conforme indicado pelo `KeyboardInterrupt`.)

In [8]:


print("Passo 2: Formatar os dados e colar os Post-its (LoRA)...")
dataset = load_dataset("json", data_files="dataset_protocolo_dengue.jsonl", split="train")

def formatar_prompt(exemplo):
    texto_formatado = f"""Abaixo está uma instrução médica baseada num protocolo. Responda de forma adequada.

### Instrução:
{exemplo['Instrucao']}

### Contexto:
{exemplo['Entrada']}

### Resposta:
{exemplo['Saida']}"""
    return {"text": texto_formatado + tokenizer.eos_token}

dataset_formatado = dataset.map(formatar_prompt)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

modelo_treinavel = get_peft_model(modelo, lora_config)

print("\nPasso 3: A iniciar o Maestro de Treino...")

argumentos_treino = TrainingArguments(
    output_dir="./modelo_dengue_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    num_train_epochs=3,
    warmup_steps=5,
    lr_scheduler_type="constant",
    gradient_checkpointing=True
)

trainer = SFTTrainer(
    model=modelo_treinavel,
    train_dataset=dataset_formatado,
    processing_class=tokenizer,
    args=argumentos_treino,
)

print("🔥 A treinar o assistente médico...")
trainer.train()

print("\n🏆 Treino concluído com sucesso! Pode guardar o seu modelo.")

Passo 2: Formatar os dados e colar os Post-its (LoRA)...

Passo 3: A iniciar o Maestro de Treino...


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


🔥 A treinar o assistente médico...


Step,Training Loss
1,1.461034
2,1.143245
3,1.348127
4,1.278530
5,1.122688
6,1.209154
7,1.236370
8,1.130270
9,1.082583
10,1.278109



🏆 Treino concluído com sucesso! Pode guardar o seu modelo.


Esta célula demonstra como testar o modelo com fine-tuning (ou o modelo base Llama 3, caso o fine-tuning tenha sido interrompido) em um contexto de RAG (Retrieval-Augmented Generation). Ela simula um cenário onde uma pergunta específica e um contexto relevante (recuperado do PDF) são fornecidos ao modelo. O modelo então gera uma resposta baseada nesta informação combinada, demonstrando a sua capacidade de fornecer respostas precisas e contextualizadas. Os parâmetros `max_new_tokens` e `temperature` são definidos para controlar o processo de geração.

In [7]:
print("A testar o Assistente Médico com Contexto (O Padrão RAG) 🩺")

# 1. A pergunta e o fragmento que o nosso sistema RAG encontraria no PDF
pergunta = "Qual o volume diário de hidratação oral para um adulto de 70kg com dengue Grupo A?"
contexto = "Adultos: 60 mL/kg/dia, sendo 1/3 com sais de reidratação oral e 2/3 com líquidos caseiros."

# 2. O Molde (Prompt) agora preenchido!
prompt_teste = f"""Abaixo está uma instrução médica baseada num protocolo. Responda de forma adequada.

### Instrução:
{pergunta}

### Contexto:
{contexto}

### Resposta:
"""

inputs = tokenizer(prompt_teste, return_tensors="pt").to("cuda")

# 3. Gerar a resposta (Aumentámos o fôlego para 300 tokens!)
with torch.no_grad():
    saida = modelo.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

resposta_final = tokenizer.decode(saida[0], skip_special_tokens=True)

print("\n" + "="*60)
print(resposta_final)
print("="*60)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A testar o Assistente Médico com Contexto (O Padrão RAG) 🩺


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Abaixo está uma instrução médica baseada num protocolo. Responda de forma adequada.

### Instrução:
Qual o volume diário de hidratação oral para um adulto de 70kg com dengue Grupo A?

### Contexto:
Adultos: 60 mL/kg/dia, sendo 1/3 com sais de reidratação oral e 2/3 com líquidos caseiros.

### Resposta:
60 mL/kg/dia = 4200 mL/dia (para um adulto de 70kg)

1/3 = 1400 mL/dia (sais de reidratação oral)
2/3 = 2800 mL/dia (líquidos caseiros)

Portanto, o volume diário de hidratação oral para um adulto de 70kg com dengue Grupo A é de 2800 mL/dia. (Líquidos caseiros) + 1400 mL/dia (sais de reidratação oral) = 4200 mL/dia. (Total) - 1400 mL/dia (sais de reidratação oral) = 2800 mL/dia. (Líquidos caseiros) + 1400 mL/dia (sais de reidratação oral) = 4200 mL/dia. (Total) - 1400 mL/dia (sais de reidratação oral) = 2800 mL/dia. (Líquidos caseiros) + 1400 mL/dia (sais de reidratação oral) = 4200 mL/dia. (Total) - 1400 mL/dia (sais de reidratação oral) = 2800 mL/dia. (Líquidos caseiros) + 1400 mL/dia